# SignSight — EDA dataset biển báo Việt Nam trên Colab
Chạy từ trên xuống; **CPU runtime là đủ**, chưa cần tốn GPU/compute units cho train.
Hai nguồn: [supsign](https://www.kaggle.com/datasets/578dd08176b0761a7a087edb6388fa15abcf21ead5fc4c490bb77a259a6d03c9)
và [data trafficsign / vitsdb](https://www.kaggle.com/datasets/ea915509b5230ca5abf2cc50f8fc31319685c45abe0471e8aa40aba5a6c6ace9).
Metadata phiên bản 1 ghi khoảng 16 GB và 14,1 GB; chưa xác nhận cấu trúc/nhãn. API public từng trả 403.
Notebook tự tải bằng KaggleHub khi tài khoản có quyền; không yêu cầu upload ZIP từ máy tính.
Có thể đọc dataset đã tải ở `LOCAL_ROOTS`, kể cả ZIP nằm trong thư mục đó, không giải nén thêm.

EDA nhận diện YOLO detection TXT, VOC XML, COCO JSON. File chưa hỗ trợ/nhãn thiếu được báo rõ,
**không coi ảnh thiếu nhãn là ảnh nền đã xác nhận**. Không train, sửa nhãn, gộp class hay xóa ảnh trùng.
Không chạy `dataset.py` hay script bất kỳ lấy từ dataset.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'kagglehub', 'Pillow', 'PyYAML', 'pandas', 'matplotlib', 'tqdm'])
from pathlib import Path
from datetime import datetime, timezone
import json, os, shutil, importlib.util, importlib.metadata, random
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 25)

## 1. Cấu hình
Giữ `MAX_IMAGES=None` để thống kê toàn bộ ảnh/nhãn; đặt `3000` nếu muốn khảo sát nhanh trước.
`HASH_LIMIT=2000` chỉ đọc toàn bộ byte của tối đa 2.000 ảnh **mỗi dataset** để tìm trùng tuyệt đối.
Đặt `None` để hash toàn bộ (có thể lâu với ~30 GB), `0` để bỏ qua. Hash mẫu không chứng minh toàn bộ dataset sạch.
`VERIFY_DECODE=False` đọc header ảnh; bật True để thử giải mã toàn bộ mỗi ảnh và tìm thêm ảnh hỏng.

Drive miễn phí thường không đủ cho cả hai bộ: mặc định cache ở runtime, ZIP báo cáo tải về cuối notebook.
`USE_DRIVE=True` giữ thêm báo cáo trên Drive; `CACHE_DATASETS_ON_DRIVE=True` mới giữ cả dữ liệu tải về,
cần đủ dung lượng và có thể chậm do I/O Drive. KaggleHub cache giúp chạy lại không tải lại khi cache còn nguyên.
Nếu thiếu ổ đĩa, chạy một dataset mỗi lượt bằng `ACTIVE_DATASETS`.

In [ ]:
ACTIVE_DATASETS = ['vitsdb', 'supsign']  # có thể dùng ['vitsdb'] trước
LOCAL_ROOTS = {}  # ví dụ {'vitsdb': '/content/drive/MyDrive/datasets/vitsdb'}; đây là THƯ MỤC
USE_DRIVE = False
CACHE_DATASETS_ON_DRIVE = False
KAGGLE_LOGIN = False  # True nếu cần đăng nhập bằng prompt; không dán API token vào code/output
MAX_IMAGES = None
ANNOTATION_FORMAT = 'auto'  # auto / yolo / voc / coco; chọn rõ nếu một ảnh có nhiều format
HASH_LIMIT = 2000
VERIFY_DECODE = False
SEED = 42
SOURCES = {
    'supsign': {'handle': 'trnghiutrng/supsign/versions/1', 'metadata_bytes': 15988455171},
    'vitsdb': {'handle': 'trnghiutrng/vitsdb/versions/1', 'metadata_bytes': 14059710820},
}
assert ACTIVE_DATASETS and len(set(ACTIVE_DATASETS)) == len(ACTIVE_DATASETS)
assert set(ACTIVE_DATASETS) <= set(SOURCES)
WORK = Path('/content/signsight_eda')
WORK.mkdir(parents=True, exist_ok=True)
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
assert not CACHE_DATASETS_ON_DRIVE or USE_DRIVE, 'Bật USE_DRIVE để giữ cache trên Drive.'
PERSISTENT = Path('/content/drive/MyDrive/SignSight/eda') if USE_DRIVE else None
CACHE = PERSISTENT/'kagglehub_cache' if CACHE_DATASETS_ON_DRIVE else WORK/'kagglehub_cache'
CACHE.mkdir(parents=True, exist_ok=True)
os.environ['KAGGLEHUB_CACHE'] = str(CACHE)
OUT = WORK/('report_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
OUT.mkdir(exist_ok=False)
disk = shutil.disk_usage(CACHE)
print(f'Ổ cache còn trống: {disk.free/1e9:.1f} GB')
print(f'Tổng dung lượng metadata nguồn đã chọn: {sum(SOURCES[s]["metadata_bytes"] for s in ACTIVE_DATASETS)/1e9:.1f} GB')
print('Cần chừa thêm dung lượng tải/cache; kích thước metadata không bảo đảm kích thước ZIP/giải nén.')

## 2. Nạp code EDA có sẵn
Cell này chứa bản sao module đã kiểm thử từ repo; không cần clone repo trên Colab.
ZIP được đọc tại chỗ; ZIP lồng trong ZIP, RAR/7z/TAR chỉ được kiểm kê, chưa đọc nội dung.
Không đổi taxonomy 10 lớp của app trong bước EDA.

In [ ]:
EDA_SOURCE = '"""Read-only detection EDA, including files inside ZIPs. No training or remapping."""\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport random\nimport xml.etree.ElementTree as ET\nimport zipfile\n\nfrom PIL import Image\nimport yaml\n\nIMAGE_EXTENSIONS = {\'.jpg\', \'.jpeg\', \'.png\', \'.bmp\', \'.webp\', \'.tif\', \'.tiff\'}\n\n\nclass DatasetReader:\n    def __init__(self, root):\n        self.root = Path(root).resolve()\n        if self.root.is_file() and self.root.suffix.lower() == \'.zip\':\n            paths, base = [self.root], self.root.parent\n        elif self.root.is_dir():\n            paths, base = sorted(self.root.rglob(\'*\')), self.root\n        else:\n            raise FileNotFoundError(f\'Dataset directory or ZIP not found: {self.root}\')\n        self.entries, self.zips, self.warnings = {}, {}, []\n        for path in paths:\n            if not path.is_file() or path.is_symlink():\n                continue\n            name = path.relative_to(base).as_posix()\n            self.entries[name] = (path, None, path.stat().st_size)\n            if path.suffix.lower() == \'.zip\':\n                try:\n                    archive = zipfile.ZipFile(path)\n                    self.zips[name] = archive\n                    for item in archive.infolist():\n                        if not item.is_dir():\n                            self.entries[name + \'!/\' + item.filename] = (archive, item, item.file_size)\n                            if PurePosixPath(item.filename).suffix.lower() in {\'.zip\', \'.rar\', \'.7z\', \'.gz\', \'.tar\'}:\n                                self.warnings.append(f\'Nested archive not expanded: {name}!/{item.filename}\')\n                except (OSError, zipfile.BadZipFile) as error:\n                    self.warnings.append(f\'Cannot open archive {name}: {type(error).__name__}\')\n\n    def open(self, name):\n        source, member, _ = self.entries[name]\n        return source.open(member) if member is not None else source.open(\'rb\')\n\n    def read(self, name, max_bytes=256 * 1024 * 1024):\n        if self.entries[name][2] > max_bytes:\n            raise ValueError(f\'Annotation exceeds {max_bytes // 1024**2} MiB read limit\')\n        with self.open(name) as stream:\n            value = stream.read(max_bytes + 1)\n        if len(value) > max_bytes:\n            raise ValueError(\'Annotation read limit exceeded\')\n        return value\n\n    def close(self):\n        for archive in self.zips.values():\n            archive.close()\n\n    def __enter__(self):\n        return self\n\n    def __exit__(self, *_):\n        self.close()\n\n\ndef split_name(name):\n    mapping = {\'train\': \'train\', \'training\': \'train\', \'val\': \'val\', \'valid\': \'val\',\n               \'validation\': \'val\', \'test\': \'test\', \'testing\': \'test\'}\n    found = {mapping[part.casefold()] for part in PurePosixPath(name).parts[:-1] if part.casefold() in mapping}\n    return next(iter(found)) if len(found) == 1 else \'unknown\'\n\n\ndef paired_files(name, extension, directory_pairs, case_names):\n    path = PurePosixPath(name)\n    candidates = {str(path.with_suffix(extension))}\n    parts = list(path.parts)\n    for i, part in enumerate(parts[:-1]):\n        for before, after in directory_pairs:\n            if part.casefold() == before.casefold():\n                changed = parts.copy(); changed[i] = after\n                candidates.add(str(PurePosixPath(*changed).with_suffix(extension)))\n    return sorted({actual for candidate in candidates for actual in case_names.get(candidate.casefold(), [])})\n\n\ndef audit_dataset(reader, dataset, image_limit=None, hash_limit=2000, seed=42,\n                  verify_decode=False, progress=None, annotation_format=\'auto\'):\n    """All counts refer to scanned images; partial/hash coverage is explicit.\n\n    Zero objects means empty_annotation only for a recognized annotation file.\n    Missing/unreadable/ambiguous labels never become verified background images.\n    """\n    if annotation_format not in {\'auto\', \'yolo\', \'voc\', \'coco\'}:\n        raise ValueError(\'annotation format must be auto, yolo, voc or coco\')\n    rng = random.Random(seed)\n    images = sorted(name for name in reader.entries if PurePosixPath(name).suffix.lower() in IMAGE_EXTENSIONS)\n    if (image_limit is not None and image_limit < 0) or (hash_limit is not None and hash_limit < 0):\n        raise ValueError(\'Limits must be nonnegative or None\')\n    selected = images if image_limit is None else sorted(rng.sample(images, min(image_limit, len(images))))\n    hashed = set(selected if hash_limit is None else rng.sample(selected, min(hash_limit, len(selected))))\n    issues, image_rows, boxes, catalogs = [], [], [], []\n    case_names = defaultdict(list)\n    for name in reader.entries:\n        case_names[name.casefold()].append(name)\n\n    def issue(path, message):\n        issues.append({\'dataset\': dataset, \'path\': path, \'message\': str(message)[:600]})\n\n    for warning in reader.warnings:\n        issue(\'\', warning)\n    for name in reader.entries:\n        suffix = PurePosixPath(name).suffix.lower()\n        try:\n            names = None\n            if suffix in {\'.yaml\', \'.yml\'}:\n                config = yaml.safe_load(reader.read(name, 1024**2))\n                if isinstance(config, dict) and \'names\' in config:\n                    names = config[\'names\']\n            elif PurePosixPath(name).name.casefold() in {\'classes.txt\', \'obj.names\', \'classes.names\'}:\n                names = reader.read(name, 1024**2).decode(\'utf-8-sig\').splitlines()\n            if names is not None:\n                if isinstance(names, list):\n                    names = dict(enumerate(names))\n                if not isinstance(names, dict) or not names:\n                    raise ValueError(\'Invalid class names\')\n                normalized = {int(key): str(value).strip() for key, value in names.items()}\n                if any(key < 0 or not value for key, value in normalized.items()):\n                    raise ValueError(\'Invalid class ID/name\')\n                catalogs.append((name, normalized))\n        except Exception as error:\n            issue(name, f\'Class metadata: {type(error).__name__}: {error}\')\n\n    def catalog_for(image_name):\n        matches = [(name, names) for name, names in catalogs\n                   if str(PurePosixPath(name).parent) == \'.\'\n                   or image_name.startswith(str(PurePosixPath(name).parent) + \'/\')]\n        if matches:\n            depth = max(len(PurePosixPath(name).parent.parts) for name, _ in matches)\n            closest = [(name, names) for name, names in matches if len(PurePosixPath(name).parent.parts) == depth]\n            if len({json.dumps(names, sort_keys=True) for _, names in closest}) > 1:\n                issue(image_name, \'Conflicting class maps at the same directory; class IDs remain unmapped.\')\n                return (\'ambiguous-class-map\', {})\n            return closest[0]\n        unique = {json.dumps(names, sort_keys=True) for _, names in catalogs}\n        return catalogs[0] if len(unique) == 1 else (\'unmapped\', {})\n\n    # COCO indexing: resolve full/suffix names; ambiguous duplicate basenames are errors.\n    by_basename = defaultdict(list)\n    for name in images:\n        by_basename[PurePosixPath(name).name].append(name)\n    coco = defaultdict(list)\n    for name in reader.entries:\n        if annotation_format not in {\'auto\', \'coco\'} or PurePosixPath(name).suffix.lower() != \'.json\':\n            continue\n        try:\n            content = json.loads(reader.read(name))\n            if not isinstance(content, dict) or not {\'images\', \'annotations\', \'categories\'} <= content.keys():\n                issue(name, \'JSON is not COCO detection; inspect separately (e.g. LabelMe/classification).\')\n                continue\n            categories = {row[\'id\']: str(row[\'name\']) for row in content[\'categories\']}\n            annotations = defaultdict(list)\n            for annotation in content[\'annotations\']:\n                annotations[annotation[\'image_id\']].append(annotation)\n            for image in content[\'images\']:\n                filename = str(image[\'file_name\']).replace(\'\\\\\', \'/\')\n                candidates = by_basename[PurePosixPath(filename).name]\n                suffix_matches = [key for key in candidates if key == filename or key.endswith(\'/\' + filename)]\n                if suffix_matches:\n                    candidates = suffix_matches\n                # Do not join a JSON in one archive to same-named images in another.\n                container = name.split(\'!/\', 1)[0] if \'!/\' in name else None\n                candidates = [key for key in candidates if\n                              (key.startswith(container + \'!/\') if container else \'!/\' not in key)]\n                if len(candidates) != 1:\n                    issue(name, f\'COCO image reference missing/ambiguous: {filename}\')\n                    continue\n                coco[candidates[0]].append((name, categories, image, annotations[image[\'id\']]))\n        except Exception as error:\n            issue(name, f\'JSON: {type(error).__name__}: {error}\')\n\n    iterator = progress(selected, desc=dataset, unit=\'image\') if progress else selected\n    for name in iterator:\n        row = {\'dataset\': dataset, \'path\': name, \'split\': split_name(name), \'parent_folder\': str(PurePosixPath(name).parent),\n               \'width\': None, \'height\': None, \'mode\': None, \'format\': None,\n               \'annotation_status\': \'missing_annotation\', \'boxes\': 0, \'sha256\': None}\n        image_rows.append(row)\n        try:\n            with reader.open(name) as stream, Image.open(stream) as image:\n                row.update(width=image.width, height=image.height, mode=image.mode)\n                if verify_decode:\n                    image.load()\n            if name in hashed:\n                digest = hashlib.sha256()\n                with reader.open(name) as stream:\n                    for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n                        digest.update(block)\n                row[\'sha256\'] = digest.hexdigest()\n        except Exception as error:\n            row[\'annotation_status\'] = \'unreadable_image\'\n            issue(name, f\'Image: {type(error).__name__}: {error}\')\n            continue\n        txt = paired_files(name, \'.txt\', [(\'images\', \'labels\')], case_names)\n        xml = paired_files(name, \'.xml\', [(\'JPEGImages\', \'Annotations\'), (\'images\', \'annotations\')], case_names)\n        options = [(\'yolo\', path) for path in txt] + [(\'voc\', path) for path in xml] + [(\'coco\', entry) for entry in coco[name]]\n        if annotation_format != \'auto\':\n            options = [option for option in options if option[0] == annotation_format]\n        if not options:\n            continue\n        if len(options) > 1:\n            row[\'annotation_status\'] = \'ambiguous_annotation\'\n            issue(name, \'Multiple matching annotations/formats; select a single export/root before training.\')\n            continue\n        kind, annotation = options[0]\n        row[\'format\'] = kind\n        raw_boxes, bad = [], False\n        try:\n            if kind == \'yolo\':\n                source, names = catalog_for(name)\n                for line_number, line in enumerate(reader.read(annotation, 8 * 1024**2).decode(\'utf-8-sig\').splitlines(), 1):\n                    if not line.strip():\n                        continue\n                    try:\n                        fields = line.split()\n                        if len(fields) != 5:\n                            raise ValueError(\'YOLO detection expects 5 columns; segmentation/other format is not inferred\')\n                        class_id = int(fields[0])\n                        if class_id < 0 or names and class_id not in names:\n                            raise ValueError(\'Class ID outside class map\')\n                        cx, cy, width, height = map(float, fields[1:])\n                        raw_boxes.append((str(class_id), names.get(class_id, f\'class_{class_id}\'), bool(names),\n                                          source, cx - width/2, cy - height/2, width, height, 0))\n                    except Exception as error:\n                        bad = True; issue(annotation, f\'Line {line_number}: {error}\')\n            elif kind == \'voc\':\n                tree = ET.fromstring(reader.read(annotation, 8 * 1024**2))\n                if tree.tag != \'annotation\':\n                    raise ValueError(\'Not a VOC annotation\')\n                for dimension in (\'width\', \'height\'):\n                    value = tree.findtext(\'size/\' + dimension)\n                    if value is not None and int(value) != row[dimension]:\n                        raise ValueError(\'VOC annotation size differs from image\')\n                for obj in tree.findall(\'object\'):\n                    label = obj.findtext(\'name\')\n                    if not label:\n                        raise ValueError(\'VOC object without class name\')\n                    xmin, ymin, xmax, ymax = [float(obj.findtext(\'bndbox/\' + key)) for key in (\'xmin\', \'ymin\', \'xmax\', \'ymax\')]\n                    raw_boxes.append((label, label, True, \'voc:names\', xmin/row[\'width\'], ymin/row[\'height\'],\n                                      (xmax-xmin)/row[\'width\'], (ymax-ymin)/row[\'height\'], 0))\n            else:\n                annotation_path, categories, image, objects = annotation\n                for dimension in (\'width\', \'height\'):\n                    if dimension in image and int(image[dimension]) != row[dimension]:\n                        raise ValueError(\'COCO annotation size differs from image\')\n                for obj in objects:\n                    class_id = obj[\'category_id\']\n                    if class_id not in categories:\n                        raise ValueError(\'COCO category missing from categories\')\n                    x, y, width, height = map(float, obj[\'bbox\'])\n                    raw_boxes.append((str(class_id), categories[class_id], True, annotation_path,\n                                      x/row[\'width\'], y/row[\'height\'], width/row[\'width\'], height/row[\'height\'], int(obj.get(\'iscrowd\', 0))))\n            for class_id, label, mapped, schema, x, y, width, height, crowd in raw_boxes:\n                if not all(math.isfinite(v) for v in (x, y, width, height)) or min(width, height) <= 0 or min(x, y) < -1e-6 or max(x+width, y+height) > 1+1e-6:\n                    bad = True; issue(name, \'Invalid bbox: nonfinite, empty, negative, or outside image\'); continue\n                boxes.append({\'dataset\': dataset, \'path\': name, \'split\': row[\'split\'], \'format\': kind,\n                              \'label_namespace\': schema, \'class_id\': class_id, \'class_name\': label, \'class_mapped\': mapped,\n                              \'x_norm\': x, \'y_norm\': y, \'width_norm\': width, \'height_norm\': height,\n                              \'area_fraction\': width * height, \'iscrowd\': crowd,\n                              \'short_side_at_640\': min(width*row[\'width\'], height*row[\'height\']) * 640 / max(row[\'width\'], row[\'height\'])})\n                row[\'boxes\'] += 1\n            row[\'annotation_status\'] = \'invalid_annotation\' if bad else \'annotated\' if row[\'boxes\'] else \'empty_annotation\'\n        except Exception as error:\n            row[\'annotation_status\'] = \'invalid_annotation\'\n            issue(name, f\'{kind}: {type(error).__name__}: {error}\')\n\n    summary = {\'dataset\': dataset, \'root\': str(reader.root), \'images_total\': len(images), \'images_scanned\': len(image_rows),\n               \'partial_scan\': len(selected) < len(images), \'images_hashed\': sum(bool(row[\'sha256\']) for row in image_rows),\n               \'hash_partial\': sum(bool(row[\'sha256\']) for row in image_rows) < len(images),\n               \'verify_decode\': verify_decode, \'annotation_format\': annotation_format,\n               \'valid_boxes\': len(boxes), \'issues\': len(issues),\n               \'invalid_annotations\': sum(row[\'annotation_status\'] == \'invalid_annotation\' for row in image_rows),\n               \'annotation_status\': dict(Counter(row[\'annotation_status\'] for row in image_rows)),\n               \'formats\': dict(Counter(row[\'format\'] for row in image_rows if row[\'format\'])),\n               \'file_extensions\': dict(Counter(PurePosixPath(name).suffix.lower() for name in reader.entries)),\n               \'class_maps\': [{\'path\': name, \'names\': names} for name, names in catalogs]}\n    return {\'summary\': summary, \'images\': image_rows, \'boxes\': boxes, \'issues\': issues}\n\n\ndef duplicate_groups(image_rows):\n    grouped = defaultdict(list)\n    for row in image_rows:\n        if row.get(\'sha256\'):\n            grouped[row[\'sha256\']].append(row)\n    return [{\'sha256\': digest, \'copies\': len(rows), \'cross_dataset\': len({r[\'dataset\'] for r in rows}) > 1,\n             \'cross_split\': len({r[\'split\'] for r in rows} - {\'unknown\'}) > 1,\n             \'members\': [{\'dataset\': r[\'dataset\'], \'path\': r[\'path\'], \'split\': r[\'split\']} for r in rows]}\n            for digest, rows in grouped.items() if len(rows) > 1]\n'
module_path = WORK/'dataset_eda.py'
module_path.write_text(EDA_SOURCE, encoding='utf-8')
spec = importlib.util.spec_from_file_location('signsight_dataset_eda', module_path)
eda = importlib.util.module_from_spec(spec)
spec.loader.exec_module(eda)
print('EDA reader sẵn sàng — không train model.')

## 3. Lấy dữ liệu
KaggleHub tự dùng Colab Secret `KAGGLE_API_TOKEN` nếu đã cấu hình, hoặc bật `KAGGLE_LOGIN` để nhập qua prompt.
Tài khoản cần được phép truy cập hai bộ; API token không tự cấp quyền dataset riêng.
Nếu 403, mở link thầy gửi bằng đúng tài khoản Kaggle và kiểm tra quyền với người chia sẻ.
Có thể tải bằng giao diện Kaggle rồi đặt thư mục chứa dữ liệu/ZIP vào `LOCAL_ROOTS`.
Không lặp retry hoặc dùng URL khác để vượt quyền truy cập. Nếu một bộ chưa tải được, báo cáo ghi rõ phạm vi bộ còn lại.
[Tài liệu KaggleHub](https://github.com/Kaggle/kagglehub).

In [ ]:
import kagglehub
if KAGGLE_LOGIN:
    kagglehub.login()
ROOTS, SOURCE_ERRORS = {}, []
for name in ACTIVE_DATASETS:
    try:
        root = Path(LOCAL_ROOTS[name]) if name in LOCAL_ROOTS else Path(kagglehub.dataset_download(SOURCES[name]['handle']))
        if not root.is_dir():
            raise FileNotFoundError('LOCAL_ROOTS phải là thư mục có dữ liệu hoặc ZIP bên trong.')
        ROOTS[name] = root
        print(name, '→', root)
    except Exception as error:
        SOURCE_ERRORS.append({'dataset':name, 'error_type':type(error).__name__})
        print(f'{name}: chưa lấy được dữ liệu ({type(error).__name__}). Kiểm tra quyền Kaggle, ổ đĩa hoặc LOCAL_ROOTS.')
if not ROOTS:
    raise RuntimeError('Chưa có nguồn đọc được. Sửa quyền/LOCAL_ROOTS rồi chạy lại cell này; chưa chạy EDA.')

## 4. Kiểm kê và đọc ảnh/nhãn
Tiến trình chỉ cập nhật một thanh mỗi dataset. Bản đồ lớp giữ riêng từng dataset và namespace file nhãn;
ID `0` của hai bộ không mặc định là cùng loại biển. Nếu nhiều bản export cùng gán nhãn một ảnh,
ảnh được đánh dấu ambiguous thay vì tự chọn một bản.
COCO JSON tối đa 256 MiB/file; YAML tên lớp tối đa 1 MiB; annotation quá giới hạn được ghi vào issues.
`split=unknown` nghĩa là chưa suy ra được từ tên thư mục, không có nghĩa dataset đã được chia đúng.

In [ ]:
for old_reader in globals().get('READERS', {}).values():
    old_reader.close()
READERS, RESULTS, INVENTORY = {}, {}, []
for name, root in ROOTS.items():
    reader = READERS[name] = eda.DatasetReader(root)
    for filename, (_, member, size) in reader.entries.items():
        INVENTORY.append({'dataset':name, 'path':filename, 'extension':Path(filename).suffix.lower(),
                          'storage_kind':'zip_member' if member is not None else 'filesystem', 'bytes':size})
    RESULTS[name] = eda.audit_dataset(reader, name, image_limit=MAX_IMAGES, hash_limit=HASH_LIMIT,
        seed=SEED, verify_decode=VERIFY_DECODE, progress=tqdm,
        annotation_format=globals().get('ANNOTATION_FORMAT', 'auto'))
    summary = RESULTS[name]['summary']
    print(f'{name}: {summary["images_scanned"]:,}/{summary["images_total"]:,} ảnh; '
          f'{summary["valid_boxes"]:,} box hợp lệ; {summary["issues"]:,} vấn đề; '
          f'{summary["images_hashed"]:,} ảnh đã hash')
inventory_df = pd.DataFrame(INVENTORY, columns=['dataset','path','extension','storage_kind','bytes'])
images_df = pd.DataFrame([r for result in RESULTS.values() for r in result['images']],
    columns=['dataset','path','split','parent_folder','width','height','mode','format','annotation_status','boxes','sha256'])
boxes_df = pd.DataFrame([r for result in RESULTS.values() for r in result['boxes']],
    columns=['dataset','path','split','format','label_namespace','class_id','class_name','class_mapped',
             'x_norm','y_norm','width_norm','height_norm','area_fraction','iscrowd','short_side_at_640'])
issues_df = pd.DataFrame([r for result in RESULTS.values() for r in result['issues']], columns=['dataset','path','message'])
duplicates = eda.duplicate_groups(images_df.to_dict('records'))
display(inventory_df.groupby(['dataset','storage_kind','extension']).size().rename('files').reset_index().head(30))
display(images_df.groupby(['dataset','split','annotation_status']).size().rename('images').reset_index())
if images_df.empty:
    print('Không thấy ảnh hỗ trợ. Xem inventory: có thể còn archive lồng/RAR/7z hoặc cấu trúc khác; chưa kết luận dataset rỗng.')
display(issues_df.head(20))

## 5. Phân bố lớp, độ phân giải, kích thước biển
Biểu đồ lớp chỉ đếm box hợp lệ đã đọc; nếu chạy sample hoặc nhiều lỗi annotation thì đây chưa phải tổng dataset.
Ảnh không có detection annotations vẫn có thể là dataset classification: xem parent_folder và inventory, không tự gán lớp từ thư mục.
`short_side_at_640` ước lượng cạnh ngắn bbox sau resize giữ tỉ lệ với cạnh dài ảnh = 640 px, trước padding;
đây không phải metric AP-small của COCO. Nhiều box rất nhỏ là lý do thử ảnh đầu vào lớn hơn trước khi chọn model xlarge.

In [ ]:
class_columns = ['dataset','label_namespace','class_id','class_name']
if not boxes_df.empty:
    class_counts = boxes_df.groupby(class_columns, dropna=False).agg(instances=('path','size'), images=('path','nunique')).reset_index()
    class_counts = class_counts.sort_values(['dataset','instances'], ascending=[True,False])
else:
    class_counts = pd.DataFrame(columns=class_columns+['instances','images'])
display(class_counts.head(30))
display(images_df[['dataset','width','height']].groupby('dataset').describe() if not images_df.empty else images_df)
display(images_df.groupby(['dataset','parent_folder']).size().rename('images').sort_values(ascending=False).head(20))
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
for name, group in images_df.dropna(subset=['width','height']).groupby('dataset'):
    axes[0,0].hist(group['width'], bins=30, alpha=.5, label=name)
    axes[0,1].scatter(group['width'], group['height'], s=5, alpha=.25, label=name)
axes[0,0].set(xlabel='Image width (px)', ylabel='Images', title='Resolution distribution')
axes[0,1].set(xlabel='Width (px)', ylabel='Height (px)', title='Image dimensions')
if not class_counts.empty:
    top = class_counts.nlargest(25, 'instances').copy()
    top['label'] = top['dataset']+' | '+top['class_id'].astype(str)+' | '+top['class_name']
    top.plot.barh(x='label', y='instances', ax=axes[1,0], legend=False)
    axes[1,0].invert_yaxis()
else:
    axes[1,0].text(.05,.5,'No readable detection boxes',transform=axes[1,0].transAxes)
axes[1,0].set_title('Top classes — dataset IDs kept separate')
for name, group in boxes_df.groupby('dataset'):
    axes[1,1].hist(group['short_side_at_640'], bins=40, alpha=.5, label=name)
axes[1,1].set(xlabel='Box short side after resize to 640 (px)', ylabel='Instances', title='Small-sign analysis')
for ax in [axes[0,0], axes[0,1], axes[1,1]]:
    if ax.get_legend_handles_labels()[0]: ax.legend()
fig.tight_layout()
fig.savefig(OUT/'distributions.png', dpi=150, bbox_inches='tight')
plt.show()
if not boxes_df.empty:
    small = boxes_df.assign(under_16px=boxes_df['short_side_at_640']<16,
                           under_32px=boxes_df['short_side_at_640']<32)
    display(small.groupby('dataset')[['under_16px','under_32px']].mean().mul(100).round(2).rename(columns=lambda x:x+'_percent'))
print('Nhóm ảnh giống hệt byte trong phạm vi hash:', len(duplicates))
print('Nhóm giao nhau giữa split đã nhận diện:', sum(group['cross_split'] for group in duplicates))
print('Nhóm giao nhau giữa dataset:', sum(group['cross_dataset'] for group in duplicates))
print('SHA-256 chỉ phát hiện file giống hệt; chưa phát hiện ảnh nén lại/frame gần trùng.')

## 6. Ảnh mẫu và bounding box
Nên xem cả ảnh nhiều box, thiếu nhãn và lỗi. Không suy diễn chất lượng dataset chỉ từ vài ảnh mẫu.
Mặc định chọn tối đa 3 ảnh mỗi dataset, ưu tiên ảnh có box; ảnh trên biểu đồ đã thu nhỏ để tránh giữ ảnh 4K trong output.

In [ ]:
chosen = []
for name, group in images_df.dropna(subset=['width','height']).groupby('dataset'):
    candidates = group.sort_values('boxes', ascending=False).head(20)
    chosen += candidates.sample(n=min(3,len(candidates)), random_state=SEED).to_dict('records')
if chosen:
    fig, axes = plt.subplots(len(chosen), 1, figsize=(14, 5*len(chosen)), squeeze=False)
    for ax, row in zip(axes.flat, chosen):
        try:
            with READERS[row['dataset']].open(row['path']) as stream, Image.open(stream) as original:
                image = original.convert('RGB')
                image.thumbnail((1400,900))
            ax.imshow(image)
            selected = boxes_df[(boxes_df['dataset']==row['dataset']) & (boxes_df['path']==row['path'])]
            for box in selected.to_dict('records'):
                x,y,w,h = box['x_norm']*image.width,box['y_norm']*image.height,box['width_norm']*image.width,box['height_norm']*image.height
                ax.add_patch(patches.Rectangle((x,y),w,h,fill=False,edgecolor='lime',linewidth=2))
                ax.text(x,y,str(box['class_name']),color='black',fontsize=8,bbox={'facecolor':'lime','alpha':.8,'pad':1})
            ax.set_title(f'{row["dataset"]} | {row["annotation_status"]} | {row["path"]}',fontsize=9)
        except Exception as error:
            ax.text(.05,.5,f'Preview failed: {type(error).__name__}',transform=ax.transAxes)
        ax.axis('off')
    fig.tight_layout();fig.savefig(OUT/'samples.png',dpi=120,bbox_inches='tight');plt.show()
else:
    print('Chưa có ảnh đọc được để vẽ. Xem issues/inventory.')

## 7. Xuất báo cáo và tải về
ZIP chỉ gồm báo cáo CSV/JSON và hình EDA, không đóng gói lại dataset ~30 GB hay API credentials.
Giữ `summary.json`, `class_counts.csv`, `issues.csv`, `duplicate_groups.json` và hình samples để quyết định bước train.
Không ghép hai bộ theo ID số trước khi đối chiếu tên lớp/QCVN. Cần kiểm tra frame/cảnh gần trùng và giữ test độc lập theo địa điểm/video.
Khuyến nghị model phải dựa trên nhãn hợp lệ, kích thước biển, số lớp và VRAM thực tế; notebook này không tự kết luận model nào tốt nhất.

In [ ]:
def export_csv(frame, name):
    # Protect spreadsheet readers while JSON keeps original class/path strings.
    safe = frame.copy()
    for column in safe.select_dtypes(include=['object','string']).columns:
        safe[column] = safe[column].map(lambda value: "'"+value if isinstance(value,str) and value.startswith(('=','+','-','@')) else value)
    safe.to_csv(OUT/name, index=False, encoding='utf-8-sig')
for filename, frame in [('inventory.csv',inventory_df),('images.csv',images_df),('boxes.csv',boxes_df),
                        ('class_counts.csv',class_counts),('issues.csv',issues_df)]:
    export_csv(frame,filename)
summary = {'created_utc':datetime.now(timezone.utc).isoformat(), 'requested_datasets':ACTIVE_DATASETS,
           'loaded_datasets':list(ROOTS), 'source_errors':SOURCE_ERRORS, 'image_limit':MAX_IMAGES,
           'hash_limit_per_dataset':HASH_LIMIT, 'seed':SEED, 'verify_decode':VERIFY_DECODE,
           'datasets':{name:result['summary'] for name,result in RESULTS.items()},
           'versions':{package:importlib.metadata.version(package) for package in ['kagglehub','Pillow','pandas','matplotlib','PyYAML']},
           'exact_duplicate_groups':len(duplicates),
           'caveats':['Missing annotation is not confirmed background.', 'Class IDs are not merged across datasets.',
                      'Exact-byte hashes do not detect re-encoded or near-duplicate images.',
                      'Charts and class counts cover scanned/readable data only.']}
(OUT/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
(OUT/'class_counts.json').write_text(class_counts.to_json(orient='records',force_ascii=False,indent=2),encoding='utf-8')
(OUT/'duplicate_groups.json').write_text(json.dumps(duplicates,ensure_ascii=False,indent=2),encoding='utf-8')
archive = Path(shutil.make_archive(str(OUT),'zip',OUT))
if PERSISTENT:
    target = PERSISTENT/'reports'/archive.name
    target.parent.mkdir(parents=True,exist_ok=True)
    assert not target.exists(), 'Không ghi đè báo cáo Drive cũ.'
    shutil.copy2(archive,target)
    print('Bản sao Drive:',target)
print('ZIP báo cáo:',archive)
from google.colab import files
files.download(str(archive))